# Improving 4CH segmentation: training recipe, cine context, more data

No decomposition here. Every condition changes how the network is trained (configs/improve.yaml):

- `legacy`: the earlier baseline recipe, re-trained so all rows are measured the same way
- `recipe`: native resolution, percentile z-score, rotation/scale/elastic + intensity augmentation (no mirroring), residual U-Net, AdamW + cosine, 80 x 250 steps
- `recipe_t5`: + the two cine phases either side of each frame as extra input channels
- `t5_extra`: + the 15 annotated 4CH_VAL cases in training
- `t5_full`: + the 16 validation cases in training
- `smp_r34_t3`: ImageNet-pretrained ResNet-34 U-Net, 3 phases as RGB

Each condition runs at seeds 42, 43, 44, and the three seed models are also tested as one ensemble. Test set: the same 15 cases as before. Dice is computed on the native pixel grid, pooled over all frames of a case, then averaged over cases; `fg_lcc` keeps only the largest blob per class.

Needs a **GPU**, **Internet**, and two datasets: the raw MRI (`cmr-multi-cine-4ch`, contains 4CH_TR) and `cmr-multi-cine-4ch-val` (contains 4CH_VAL). Without the second, `t5_extra` and `t5_full` are skipped.

In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir(), "Run this notebook on Kaggle."

REPO = WORK / "cardiac-mri-4ch-segmentation"
COMMIT = "SET_AFTER_PUSH"  # commit containing configs/improve.yaml
URL = "https://github.com/sancia-jw/cardiac-mri-4ch-segmentation.git"
assert COMMIT != "SET_AFTER_PUSH", "Pin COMMIT to the pushed commit first."

if not REPO.exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, f"Existing checkout is {head}; expected {COMMIT}."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel", "PyYAML", "segmentation-models-pytorch"], check=True)
print("Repository and dependencies ready:", head)

Locate the data. DATA must contain 4CH_TR; EXTRA must contain 4CH_VAL (optional).

In [ ]:
import os

DATA_OVERRIDE = None   # e.g. "/kaggle/input/cmr-multi-cine-4ch/CMR-MULTI/CINE_MULTI"
EXTRA_OVERRIDE = None  # e.g. "/kaggle/input/cmr-multi-cine-4ch-val"

def roots_with(view):
    # os.walk(followlinks=True): Kaggle can mount datasets as symlinks, which Path.rglob skips.
    return sorted({Path(root) for root, dirs, _ in os.walk(INPUT, followlinks=True)
                   if view in dirs and (Path(root) / view / "image").is_dir() and (Path(root) / view / "anno").is_dir()})

found = roots_with("4CH_TR")
DATA = Path(DATA_OVERRIDE) if DATA_OVERRIDE else (found[0] if len(found) == 1 else None)
if DATA is None:
    raise RuntimeError(f"Expected one folder containing 4CH_TR, found {found}. Attach the raw MRI dataset.")
extra = roots_with("4CH_VAL")
EXTRA = Path(EXTRA_OVERRIDE) if EXTRA_OVERRIDE else (extra[0] if extra else None)
OUT = WORK / "improve"
LOGS = WORK / "improve_logs"
CONFIG = REPO / "configs/improve.yaml"
print("DATA:", DATA)
print("EXTRA:", EXTRA or "not attached: t5_extra and t5_full will be skipped")

Check the GPU and inputs (case split, image sizes, cine loops). Nothing is trained.

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU in Kaggle notebook settings."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
LOGS.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44]
BUDGET_HOURS = 11.3  # Kaggle stops a session at 12 h; runs that would not finish are not started

def command(*extra_args):
    cmd = [sys.executable, "-u", str(REPO / "scripts/train_improve.py"), "--config", str(CONFIG),
           "--data-root", str(DATA), "--output-root", str(OUT), "--seeds", *map(str, SEEDS)]
    if EXTRA:
        cmd += ["--extra-root", str(EXTRA)]
    return cmd + list(extra_args)

def run_logged(cmd, log_name):
    log_path = LOGS / log_name
    print("Running:", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
            cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f"Command exited with code {code}; see {log_path}")

run_logged(command("--validate-only"), "validation.log")

**Training.** Finished runs are skipped, so re-running continues where a session stopped.

In [ ]:
run_logged(command("--budget-hours", str(BUDGET_HOURS)), "training.log")

Results. `fg` is the mean over seeds; `delta` is the condition minus `legacy` at the same seed, averaged; `rel_%` is that as a percentage of `legacy`. The `ensemble` rows average the three seed models.

In [ ]:
import pandas as pd
from IPython.display import display

s = pd.read_csv(OUT / "summary.csv", dtype={"seed": str})
names = {"dice_1": "LV", "dice_2": "LV_myo", "dice_3": "RV", "dice_4": "RA", "dice_5": "LA"}
seeds = s[s.seed != "ensemble"].copy()
base = seeds[seeds.condition == "legacy"].set_index("seed")["fg"]
seeds["delta"] = seeds["fg"] - seeds["seed"].map(base)
order = list(dict.fromkeys(s.condition))
table = seeds.groupby("condition").agg(seeds=("seed", "count"), fg=("fg", "mean"), fg_sd=("fg", "std"),
                                       fg_lcc=("fg_lcc", "mean"), legacy_metric=("legacy_fg", "mean"),
                                       delta=("delta", "mean"), delta_sd=("delta", "std"),
                                       **{v: (k, "mean") for k, v in names.items()}).reindex(order)
table["rel_%"] = 100 * table["delta"] / base.mean()
ens = s[s.seed == "ensemble"].set_index("condition")[["fg", "fg_lcc", *names]].rename(columns=names)
ens["rel_%_vs_legacy"] = 100 * (ens["fg"] - base.mean()) / base.mean()
table.round(4).to_csv(OUT / "seed_table.csv")
ens.round(4).to_csv(OUT / "ensemble_table.csv")
display(table.round(4))
display(ens.round(4))